# 5단계 · 최종 평가 (테스트셋)

지금까지 **한 번도 사용하지 않은 테스트셋(141개)**으로 세 모델의 최종 성적을 확정합니다. 테스트셋은 학습·튜닝에 전혀 쓰지 않았기에, 여기 점수가 **진짜 실력**에 가장 가깝습니다.

비교할 모델 3개:
1. 다수 클래스 (항상 of)
2. TF-IDF + 로지스틱 회귀
3. **DistilBERT (4단계에서 파인튜닝해 저장한 모델)**

마지막에 결과를 `results/` 폴더에 저장해 README/포트폴리오에서 인용합니다.

In [1]:
import pandas as pd, numpy as np, torch
from pathlib import Path
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import pipeline   # 저장된 모델을 추론에 간편히 쓰는 도구

# 베이스라인은 학습셋으로 다시 학습해야 하므로 train도 불러옵니다. test는 최종 평가용.
train_df = pd.read_csv("../data/splits/train.csv")
test_df  = pd.read_csv("../data/splits/test.csv")
print("학습:", len(train_df), "테스트:", len(test_df))

/Users/jamie/anaconda3/envs/default/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


학습: 657 테스트: 141


In [2]:
# 입력(가려진 문장)과 정답(전치사) 준비 — 4단계와 동일한 입력 형태로 공정하게 비교
X_train, y_train = train_df["sentence_masked"], train_df["label"]
X_test,  y_test  = test_df["sentence_masked"],  test_df["label"]
labels = ["of", "from", "with"]   # 표/혼동행렬의 순서 고정

## 모델 ①② 베이스라인 — 테스트셋 점수

In [3]:
# ① 다수 클래스: 학습셋에서 가장 흔한 클래스를 항상 예측
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
pred_dummy = dummy.predict(X_test)

# ② TF-IDF + 로지스틱 회귀 (3단계와 동일 설정)
logreg = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2)),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
]).fit(X_train, y_train)
pred_lr = logreg.predict(X_test)

print("베이스라인 학습/예측 완료")

베이스라인 학습/예측 완료


## 모델 ③ DistilBERT — 테스트셋 점수

4단계에서 `outputs/distilbert-best/`에 저장한 모델을 불러와 테스트 문장을 예측합니다.

In [4]:
# 저장된 모델+토크나이저를 추론 파이프라인으로 로드. M4 가속(MPS) 사용.
device = "mps" if torch.backends.mps.is_available() else "cpu"
clf = pipeline(
    "text-classification",
    model="../outputs/distilbert-best",
    tokenizer="../outputs/distilbert-best",
    truncation=True, max_length=128,   # 학습 때와 동일하게 긴 문장은 128토큰으로 자름
    device=device,
)
# 테스트 문장 전체를 예측. 결과의 'label'이 of/from/with 글자로 나옵니다(모델에 저장된 사전 덕분).
pred_bert = [r["label"] for r in clf(X_test.tolist(), batch_size=32)]
print("DistilBERT 예측 완료. 예시:", pred_bert[:5])

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 10210.14it/s]

DistilBERT 예측 완료. 예시: ['of', 'with', 'of', 'of', 'of']


## 최종 비교표 (테스트셋)

In [5]:
# 세 모델의 정확도와 Macro-F1을 한 표로 모읍니다.
def score(name, pred):
    return {
        "model": name,
        "accuracy": round(accuracy_score(y_test, pred), 4),
        "macro_f1": round(f1_score(y_test, pred, average="macro"), 4),
    }

results = pd.DataFrame([
    score("① 다수 클래스", pred_dummy),
    score("② TF-IDF + LogReg", pred_lr),
    score("③ DistilBERT (파인튜닝)", pred_bert),
])
results

,model,accuracy,macro_f1
0,① 다수 클래스,0.5461,0.2355
1,② TF-IDF + LogReg,0.4539,0.3787
2,③ DistilBERT (파인튜닝),0.6099,0.5746


In [6]:
# DistilBERT의 클래스별 상세 성적 (테스트셋)
print(classification_report(y_test, pred_bert, labels=labels, digits=3))

              precision    recall  f1-score   support

          of      0.785     0.662     0.718        77
        from      0.444     0.432     0.438        37
        with      0.475     0.704     0.567        27

    accuracy                          0.610       141
   macro avg      0.568     0.599     0.575       141
weighted avg      0.636     0.610     0.616       141



In [7]:
# DistilBERT 혼동행렬 (테스트셋): 행=실제, 열=예측
cm = confusion_matrix(y_test, pred_bert, labels=labels)
cm_df = pd.DataFrame(cm, index=[f"실제 {l}" for l in labels], columns=[f"예측 {l}" for l in labels])
cm_df

,예측 of,예측 from,예측 with
실제 of,51,14,12
실제 from,12,16,9
실제 with,2,6,19


## 논문과의 비교 및 해석

- 석사 논문에서 **GPT-4o(무학습)**는 전체 정확도 **39%**, 특히 `from`은 **0%**였습니다.
- 우리 DistilBERT는 데이터로 학습한 결과 베이스라인을 넘었고, `from`도 0%에서 크게 개선됐습니다.
- 주의: 논문 39%는 데이터 분할·입력 방식이 달라 **1:1 직접 비교는 아니며**, '같은 문제를 학습으로 풀면 얼마나 나아지는가'를 보여주는 *동기 부여용* 수치입니다.

아래에서 결과를 `results/` 폴더에 저장합니다.

In [8]:
# 결과 파일을 results/ 폴더에 저장 (README/포트폴리오에서 인용). 작은 파일이라 깃에 포함해도 됩니다.
out = Path("../results"); out.mkdir(exist_ok=True)
results.to_csv(out / "test_results.csv", index=False)                 # 모델별 정확도/Macro-F1
cm_df.to_csv(out / "distilbert_confusion_matrix.csv")                 # 혼동행렬
with open(out / "distilbert_classification_report.txt", "w") as f:    # 클래스별 상세표
    f.write(classification_report(y_test, pred_bert, labels=labels, digits=3))
print("저장 완료 ->", out.resolve())
print("\n[최종 요약]")
print(results.to_string(index=False))

저장 완료 -> /Users/jamie/Desktop/00_github_repositories/myproject/results

[최종 요약]
              model  accuracy  macro_f1
           ① 다수 클래스    0.5461    0.2355
  ② TF-IDF + LogReg    0.4539    0.3787
③ DistilBERT (파인튜닝)    0.6099    0.5746
